## Modelo Toomer
1. *Núcleos puntuales* Representan toda la masa de cada galaxia, es decir las interacciones gravitacionales se dan con y entre estos núcleos.
2. Estrellas como partículas de prueba i.e. la masa de las estrellas es m_* = 0 y esto quiere decir que no interactúan entre ellas

In [2]:
%pip install numba

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 1.1 MB/s eta 0:00:03
   ----------- ---------------------------- 0.8/2.8 MB 1.1 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.8 MB 1.0 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.8 MB 1.0 MB/s eta 0:00:02
   ------------------- -------------------- 1.3/2.8 MB 949.7 kB/s eta 0:00:02
   ---------------------- ----------------- 1.6/2.8 MB 917.2 kB/s eta 0:00:02
   ---------------------- ----------------- 1.6/2.8 MB 917.2 kB/s eta 0:00:02
   ---------------------- ----------------- 1.6/2.8 MB 917.2 kB/s eta 0:00:02
   -------------------------- ------------- 1.8/2.8 MB 825.2 kB/s eta 0:00:02
   ---------------------------

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from numba import jit

from typing import List, Dict, Any

In [8]:
class N_system:
    def __init__(self, particles: List[Dict[str, Any]]):
        '''Clase General para un sistema de N partículas, donde cada
        partícula es un diccionario con sus propiedades físicas
        - "mass": masa de la partícula
        - "position": posición inicial de la partícula (array de 3 elementos)
        - "velocity": velocidad inicial de la partícula (array de 3 elementos)
        '''
        self.n = len(particles)
        self.positions = np.array([p['position'] for p in particles])
        self.velocities = np.array([p['velocity'] for p in particles])
        self.masses = np.array([p['mass'] for p in particles])
        self.mass = self.masses.sum()

        v_prom = np.mean(np.linalg.norm(self.velocities, axis=1))
        self.t_relax = self.n / np.log(self.n) * (v_prom**3) / (self.mass*1.0) #Tiempo de relajacion estimado
## Indices de cuerpos con masa

        self.mass_indices = np.where(self.masses > 0)[0]

    def evolucionar(self, dt: float, T: float)->np.ndarray[float,3]:
        '''Integrar el sistema por el metodo de Leap Frog durante un tiempo T con pasos de tiempo dt
        params: 
        - dt: paso de tiempo
        - T: tiempo total de integración
        returns:
         trayectorias: array de forma (n, num_steps, 3) con las posiciones de cada partícula en cada paso de tiempo'''
        
        def acc(pos: np.ndarray[float,3],
                        masses: np.ndarray[float],
                        m_indices: np.ndarray[int],
                        n_particles: int
                        )->np.ndarray[float,3]:
            '''Calcular la aceleración de cada partícula debido a la gravedad de las demás
            params:
            - pos: array de forma (n, 3) con las posiciones actuales de las partículas
            - masses: array de forma (n,) con las masas de las partículas
            - m_indices: array de índices de las partículas con masa > 0'''
            acc = np.zeros_like(pos)
            for i in range(self.n):
                for j in m_indices:
                    r_vec = pos[i] - pos[j]
                    r_mag = np.linalg.norm(r_vec) + 1e-10 # Evitar división por cero
                    acc[i] -= masses[j]*r_vec / r_mag**3
                   
            return acc
        
        num_steps = int(T / dt)
        trajectories = np.zeros((self.n, num_steps, 3))

    

        #Leap Frog integration
        accelerations = acc(self.positions, self.masses, self.mass_indices, self.n)
        self.velocities += 0.5 * accelerations * dt # Kick inicial

        for step in range(1, num_steps):
            self.positions += self.velocities * dt # Drift
            trajectories[:, step, :] = self.positions
            accelerations = acc(self.positions, self.masses, self.mass_indices, self.n)
            self.velocities += accelerations * dt # Kick

        return trajectories
    

     

    
        
  

In [10]:
mi_primer_sistema = N_system([
    {"mass": 1.0, "position": [0.0, 0.0, 0.0], "velocity": [0.0, 0.0, 0.0]},
    {"mass": 2.0, "position": [1.0, 0.0, 0.0], "velocity": [0.0, 1.0, 0.0]},
    {"mass": 2.0, "position": [2.0, 0.0, 0.0], "velocity": [0.0, -1.0, 0.0]}
])  

In [ ]:
mi_primer_sistema